#### Notebook to train a custom encoder model on the pii ds from scratch.
*This model won't be optimized with hyperparameters. It's purpose is just to set a baseline for a fine-tuned bert model to compare to.*

In [ ]:
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset

from tqdm import tqdm
import wandb
import mlflow

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Implements sinusoidal positional encoding to provide sequence order 
    information to the model.
    """
    def __init__(self, max_len: int, d_model: int) -> None:
        """init sinusoidal positional encoding

        Args:
            max_len (int): max length of token sequence
            d_model (int): embedding length
        """
        super().__init__()
        self.max_len = max_len
        self.d_model = d_model
        
        # initialize tensor to hold positional encodings
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        
        # Calculate the scaling factor for the wavelengths of the sinusoidal functions
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             -(torch.log(torch.tensor(10_000.0)) / d_model))
        
        # sin for even, cos for odd
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # save pe with register_buffer so gradients wont be computed
        self.register_buffer("pe", pe.unsqueeze(0))
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        # slice positional encoding to match input tensor
        return self.pe[:, :seq_len, :] # type: ignore

In [ ]:
class EmbeddingsBlock(nn.Module):
    """Embedding Block for a transformer model. 
    It combines positional sinusoidal encoding and token embedding to produce
    a complex representation for every word in a sequence
    """
    def __init__(self, max_seq_len: int, embed_dim: int, vocab_size: int,
                 drop_p: float, pad_idx: int=0):
        """

        Args:
            max_seq_len (int): max length of sequence passed to the model
            embed_dim (int): desired dimensions for embedding
            vocab_size (int): size of tokenizer's vocab
            drop_p (float): probability fro dropout
            pad_idx (int, optional): tokenizer's index for padding. Defaults to 0.
        """
        super().__init__()
        self.pe = PositionalEncoding(max_seq_len, embed_dim)
        self.te = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.ln = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(drop_p)
        
    def forward(self, x):
        # add position encoding to token embedding
        emb = self.pe(x) + self.te(x)
        # normalize
        emb = self.ln(emb)
        # return with dropout
        return self.dropout(emb)

In [ ]:
class CustomEncoder(nn.Module):
    """A Custom Transformer Encoder model for NER that uses positional 
    and token embeddings, transformer encoder blocks with batch and norm first,
    and a final dense classifier layer. 
    """
    def __init__(self, max_len:int, d_model:int, vocab_size: int, dropout_p: float, 
                 nhead:int, ff_dim: int, num_layers: int, num_classes: int, 
                 pad_idx: int=0):
        """

        Args:
            max_len (int): max length of sequence passed to model (for positional encoding)
            d_model (int): embedding dim
            vocab_size (int): size of tokenizer's vocab
            dropout_p (float): dropout probability (for embeddings and encoders)
            nhead (int): number of heads per encoder layer
            ff_dim (int): feedforward dimensions for dense layers in encoder block
            num_layers (int): number of encoder layers
            num_classes (int): number of classes model will classify
            pad_idx (int, optional): tokenizer's padding index. Defaults to 0.
        """
        super().__init__()
        self.embeddings_block = EmbeddingsBlock(max_seq_len=max_len, embed_dim=d_model, 
                                          vocab_size=vocab_size, drop_p=dropout_p,
                                          pad_idx=pad_idx)
        # store padding index for padding mask
        self.pad_idx = pad_idx
        
        # norm first to not require warmup like bert did
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=ff_dim, 
            dropout=dropout_p, batch_first=True, norm_first=True)
        
        self.transformer_blocks = nn.TransformerEncoder(encoder_layer, num_layers)
            
        # add a final layer norm because the encoder layers norm first
        self.ln_final = nn.LayerNorm(d_model)

        # final classifier layer
        self.classifier = nn.Linear(d_model, num_classes)
    
    def forward(self, x: torch.Tensor):
        # create padding mask for self attention to ignore
        padding_mask = x == self.pad_idx
        
        # get embeddings
        x = self.embeddings_block(x)
        
        # pass embeddings and padding mask through transformer blocks
        x = self.transformer_blocks(
            x,
            src_key_padding_mask=padding_mask
        )
        
        # apply layer norm and get final predicted logits
        x = self.ln_final(x)
        logits = self.classifier(x)
        
        return logits
        

In [ ]:
model = CustomEncoder(max_len=1_000, d_model=768, vocab_size=30_522, dropout_p=0.1, nhead=12, ff_dim=4*768, num_layers=12, num_classes=57)

CustomEncoder(
  (embeddings_block): EmbeddingsBlock(
    (pe): PositionalEncoding()
    (te): Embedding(30522, 768, padding_idx=0)
    (ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_blocks): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=3072, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=3072, out_features=768, bias=True)
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_final): LayerNorm((768,), eps

/tmp/ipykernel_16244/3482615289.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_blocks = nn.TransformerEncoder(encoder_layer, num_layers)


In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs